# Tanzania voice benchmark — Phase 3 (v2)

**EXPERIMENTAL / untested.** Hear candidate voice engines read the
`TANZANIA_VOICE_BENCHMARK.md` sentences, then score them against the §4 bar.

- **MMS-TTS (swh)** — real Swahili, one fixed speaker, *not* your voice. The
  accent/pronunciation **baseline**. Reliable — run this first.
- **XTTS v2** — clones *your* voice from a reference clip, but has **no
  Swahili**. Used only to judge *“does this sound like me?”* (timbre).

Steps: `Runtime → Change runtime type → T4 GPU`, then run the cells top to
bottom. Upload your reference clip (a short video is fine) when asked.

In [ ]:
#@title 1. GPU check
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > T4 GPU, then rerun')

In [ ]:
#@title 2. Get the tooling (clone or update), then move into it
import os, subprocess
REPO = '/content/repo'
BRANCH = 'phase-2-mock-skeleton'
URL = 'https://github.com/nyumbafasta07-sketch/nyumbafasta-video-studio-1'
if not os.path.isdir(REPO + '/.git'):
    print(subprocess.run(['git','clone','--depth','1','-b',BRANCH,URL,REPO],
                         capture_output=True, text=True).stderr)
else:
    subprocess.run(['git','-C',REPO,'pull'], check=False)
WORK = REPO + '/worker/colab'
assert os.path.isfile(WORK + '/voice_worker.py'), 'clone failed — see the message above'
os.chdir(WORK)
print('OK, working in', os.getcwd())
!ls

In [ ]:
#@title 3. Install deps for the MMS baseline (fast)
!pip -q install "transformers>=4.44" torch soundfile
print('done')

In [ ]:
#@title 4. Upload your reference clip (video or audio)
from google.colab import files
up = files.upload()
SRC = next(iter(up))
print('uploaded:', SRC)

In [ ]:
#@title 5. Make a clean reference wav + listen to it
#@markdown Pick a stretch where you're just talking, no music/noise.
START_SECONDS = 0  #@param {type:"integer"}
DURATION_SECONDS = 20  #@param {type:"integer"}
import subprocess, IPython.display as ipd
REF = '/content/reference.wav'
subprocess.run(['ffmpeg','-y','-i',SRC,'-vn','-ac','1','-ar','22050',
                '-ss',str(START_SECONDS),'-t',str(DURATION_SECONDS),REF],
               check=True, capture_output=True)
print('reference:', REF, '— does this sound like a clean clip of you?')
ipd.display(ipd.Audio(REF))

In [ ]:
#@title 6. helper to play a backend's output
import glob, json, os, IPython.display as ipd
def play(backend):
    d = f'out/{backend}'
    res = f'{d}/_results.json'
    if os.path.exists(res):
        for r in json.load(open(res)):
            print(f"[{r['id']:5s}] {r['category']:14s} {r['status']}")
    wavs = sorted(glob.glob(f'{d}/*.wav'))
    if not wavs:
        print(f'\n>>> {backend}: NO audio was produced. Read the ERROR lines above.')
        return
    for f in wavs:
        print(f); ipd.display(ipd.Audio(f))

In [ ]:
#@title 7. MMS-TTS (swh) baseline — real Swahili
!python voice_worker.py --mode benchmark --backend mms --out ./out
play('mms')

### Score MMS against `TANZANIA_VOICE_BENCHMARK.md`

Does the Swahili sound **Tanzanian**? Pronunciation of `ng'ombe`, `Mng'ong'o`,
`mchicha`? Rhythm? Robotic? This is the floor other models must beat.

---

## Optional — XTTS v2 timbre check (does it sound like *you*?)

XTTS can't speak Swahili, so `--lang en` is used. Judge only whether the voice
**colour** matches yours on the English `ID1–ID3` lines. The Swahili lines will
be mispronounced — that's expected.

In [ ]:
#@title 8. Install XTTS (slow, ~3–5 min) then generate
!pip -q install coqui-tts
!COQUI_TOS_AGREED=1 python voice_worker.py --mode benchmark --backend xtts --ref /content/reference.wav --lang en --out ./out
play('xtts')

## What to send back

1. Did cell 7 (MMS) produce audio? If not, paste its ERROR lines.
2. Your reaction to MMS Swahili (Tanzanian? robotic? pronunciation errors?).
3. Did cell 8 (XTTS) produce audio? Does `ID1–ID3` sound like you?

That decides the next candidate (F5-TTS, OpenVoice, Fish/OpenAudio) or a small
fine-tune on your recordings (brief §8).

## Optional — run the app against the real voice

Cell below serves the MMS voice over a public URL. Put it in `.env` as
`GPU_PROVIDER=http` + `GPU_WORKER_URL=<url>`. Avatar / lip-sync / render stay
mock until Phases 4–5.

In [ ]:
#@title 9. (optional) serve over a public URL
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import subprocess, threading, time
threading.Thread(target=lambda: subprocess.run(
    ['python','voice_worker.py','--mode','serve','--backend','mms','--port','8800']), daemon=True).start()
time.sleep(4)
!cloudflared tunnel --url http://localhost:8800